# Session 2: Advanced MongoDB and Data Modeling

## Data Ecosystems and Governance in Organizations
**MSc Business Analytics | Nova School of Business and Economics**

---

## Learning Objectives

By the end of this session, you will be able to:

1. 🔍 Use advanced query operators for complex data retrieval
2. ⚙️ Build aggregation pipelines for data analysis
3. 🗂️ Design effective MongoDB schemas (embedding vs. referencing)
4. ⚡ Implement indexing strategies for performance optimization
5. ✅ Apply schema validation for data quality

## Today's Agenda

1. Advanced Query Operators
2. The Aggregation Pipeline
3. Data Modeling Patterns
4. Indexing for Performance
5. Schema Validation
6. Hands-On Exercises

> ⚠️ **Reminder:** Session 3 = Quiz covering Sessions 1 & 2

---
# Setup: Load Sample Data
---

**Make sure MongoDB is running before executing these cells!**

In [2]:
from pymongo import MongoClient, ASCENDING, DESCENDING
from datetime import datetime, timedelta
from pprint import pprint
import random

# Connect
client = MongoClient('localhost', 27017)
db = client['business_analytics']

# Clear existing data
db.orders.drop()
db.customers.drop()
db.products_v2.drop()

print("✓ Connected and cleared collections")

✓ Connected and cleared collections


In [3]:
# Create products
products_data = [
    {"_id": "P001", "name": "Laptop Pro 15", "category": "Electronics", "price": 1299.99, "stock": 50},
    {"_id": "P002", "name": "Wireless Mouse", "category": "Electronics", "price": 29.99, "stock": 200},
    {"_id": "P003", "name": "USB-C Hub", "category": "Electronics", "price": 49.99, "stock": 150},
    {"_id": "P004", "name": "Mechanical Keyboard", "category": "Electronics", "price": 129.99, "stock": 75},
    {"_id": "P005", "name": "Monitor 27 inch", "category": "Electronics", "price": 399.99, "stock": 40},
    {"_id": "P006", "name": "Office Chair", "category": "Furniture", "price": 299.99, "stock": 25},
    {"_id": "P007", "name": "Standing Desk", "category": "Furniture", "price": 549.99, "stock": 15},
    {"_id": "P008", "name": "Desk Lamp", "category": "Furniture", "price": 45.99, "stock": 100},
    {"_id": "P009", "name": "Notebook Set", "category": "Office Supplies", "price": 12.99, "stock": 500},
    {"_id": "P010", "name": "Pen Pack", "category": "Office Supplies", "price": 8.99, "stock": 1000}
]

# Create customers
customers_data = [
    {"_id": "C001", "name": "John Smith", "email": "john@example.com", "city": "Lisbon", "segment": "Premium"},
    {"_id": "C002", "name": "Maria Garcia", "email": "maria@example.com", "city": "Madrid", "segment": "Standard"},
    {"_id": "C003", "name": "Hans Mueller", "email": "hans@example.com", "city": "Berlin", "segment": "Premium"},
    {"_id": "C004", "name": "Sophie Dubois", "email": "sophie@example.com", "city": "Paris", "segment": "Standard"},
    {"_id": "C005", "name": "Marco Rossi", "email": "marco@example.com", "city": "Rome", "segment": "Budget"}
]

db.products_v2.insert_many(products_data)
db.customers.insert_many(customers_data)
print(f"✓ Inserted {len(products_data)} products and {len(customers_data)} customers")

✓ Inserted 10 products and 5 customers


In [4]:
# Generate 100 sample orders
random.seed(42)  # For reproducibility
statuses = ["pending", "processing", "shipped", "delivered", "cancelled"]
regions = ["Europe", "Europe", "Europe", "Americas", "Asia"]

orders_data = []
base_date = datetime(2024, 1, 1)

for i in range(100):
    customer = random.choice(customers_data)
    items = []
    
    for _ in range(random.randint(1, 4)):
        product = random.choice(products_data)
        qty = random.randint(1, 5)
        items.append({
            "product_id": product["_id"],
            "product_name": product["name"],
            "category": product["category"],
            "quantity": qty,
            "price": product["price"],
            "subtotal": round(product["price"] * qty, 2)
        })
    
    orders_data.append({
        "_id": f"ORD{i+1:04d}",
        "customer_id": customer["_id"],
        "customer_name": customer["name"],
        "order_date": base_date + timedelta(days=random.randint(0, 180)),
        "status": random.choice(statuses),
        "region": random.choice(regions),
        "items": items,
        "total": round(sum(item["subtotal"] for item in items), 2)
    })

db.orders.insert_many(orders_data)
print(f"✓ Generated {len(orders_data)} orders")

✓ Generated 100 orders


In [5]:
# Preview the data structure
print("Sample order:")
pprint(db.orders.find_one())

Sample order:
{'_id': 'ORD0001',
 'customer_id': 'C001',
 'customer_name': 'John Smith',
 'items': [{'category': 'Electronics',
            'price': 399.99,
            'product_id': 'P005',
            'product_name': 'Monitor 27 inch',
            'quantity': 2,
            'subtotal': 799.98}],
 'order_date': datetime.datetime(2024, 2, 27, 0, 0),
 'region': 'Europe',
 'status': 'processing',
 'total': 799.98}


---
# Part 1: Advanced Query Operators
---

## Query Operator Categories

| Category | Operators | Purpose |
|----------|-----------|--------|
| **Comparison** | `$gt`, `$lt`, `$gte`, `$lte`, `$eq`, `$ne` | Compare values |
| **Set** | `$in`, `$nin` | Match against arrays |
| **Logical** | `$and`, `$or`, `$not`, `$nor` | Combine conditions |
| **Element** | `$exists`, `$type` | Check field existence/type |
| **Array** | `$elemMatch`, `$size`, `$all` | Query array contents |

**Basic syntax:** `{ field: { $operator: value } }`

## Comparison Operators

In [6]:
# $gt, $gte, $lt, $lte - Greater/Less than
print("High-value orders (> $500):")
for order in db.orders.find({"total": {"$gt": 500}}).limit(5):
    print(f"  {order['_id']}: ${order['total']:,.2f}")

High-value orders (> $500):
  ORD0001: $799.98
  ORD0003: $6,564.90
  ORD0004: $5,076.90
  ORD0005: $2,219.86
  ORD0007: $1,719.92


In [7]:
# $in - Match any value in array
shipped_delivered = db.orders.count_documents({
    "status": {"$in": ["shipped", "delivered"]}
})
print(f"Shipped or delivered: {shipped_delivered} orders")

# $nin - Not in array
active = db.orders.count_documents({
    "status": {"$nin": ["cancelled", "pending"]}
})
print(f"Active orders (not cancelled/pending): {active}")

Shipped or delivered: 40 orders
Active orders (not cancelled/pending): 55


## Logical Operators

In [8]:
# $and - All conditions must match (explicit)
query = {
    "$and": [
        {"total": {"$gte": 300}},
        {"status": "delivered"},
        {"region": "Europe"}
    ]
}
count = db.orders.count_documents(query)
print(f"High-value delivered orders in Europe: {count}")

High-value delivered orders in Europe: 6


In [9]:
# Implicit $and (cleaner syntax for different fields)
query = {
    "total": {"$gte": 300},
    "status": "delivered",
    "region": "Europe"
}
print(f"Same query, implicit $and: {db.orders.count_documents(query)}")

Same query, implicit $and: 6


In [10]:
# $or - At least one condition must match
premium_customers = ["C001", "C003"]

query = {
    "$or": [
        {"total": {"$gte": 1000}},
        {"customer_id": {"$in": premium_customers}}
    ]
}
print(f"High-value OR Premium customer: {db.orders.count_documents(query)} orders")

High-value OR Premium customer: 74 orders


## Array Operators

In [11]:
# $elemMatch - Array element meets multiple criteria
query = {
    "items": {
        "$elemMatch": {
            "category": "Electronics",
            "quantity": {"$gte": 3}
        }
    }
}

print("Orders with 3+ Electronics items in a single line:")
for order in db.orders.find(query).limit(3):
    print(f"\n{order['_id']}:")
    for item in order['items']:
        if item['category'] == 'Electronics' and item['quantity'] >= 3:
            print(f"  → {item['product_name']}: qty {item['quantity']}")

Orders with 3+ Electronics items in a single line:

ORD0003:
  → Laptop Pro 15: qty 5

ORD0005:
  → Wireless Mouse: qty 3

ORD0007:
  → Wireless Mouse: qty 4
  → Monitor 27 inch: qty 4


> ⚠️ **Common Mistake:**
> ```python
> {"items.category": "Electronics", "items.quantity": {"$gte": 3}}
> ```
> This matches if *any* item is Electronics AND *any* item has qty ≥ 3 — **not the same item!**
>
> Use `$elemMatch` when conditions must apply to the **same array element**.

In [12]:
# $size - Exact array length
three_items = db.orders.count_documents({"items": {"$size": 3}})
print(f"Orders with exactly 3 items: {three_items}")

Orders with exactly 3 items: 19


---
# Part 2: The Aggregation Pipeline
---

## What is the Aggregation Pipeline?

A framework for data transformation where documents flow through **stages**, each transforming the data — like Unix pipes.

```
Documents → [$match] → [$unwind] → [$group] → [$sort] → Results
```

### Pipeline Stages — SQL Mapping

| Stage | SQL Equivalent | Purpose |
|-------|----------------|--------|
| `$match` | `WHERE` | Filter documents |
| `$group` | `GROUP BY` | Aggregate data |
| `$project` | `SELECT` | Reshape output |
| `$sort` | `ORDER BY` | Sort results |
| `$limit`/`$skip` | `LIMIT`/`OFFSET` | Pagination |
| `$lookup` | `LEFT JOIN` | Join collections |
| `$unwind` | — | Deconstruct arrays |

## Basic Aggregation: Revenue by Status

In [13]:
pipeline = [
    # Stage 1: Group by status
    {
        "$group": {
            "_id": "$status",
            "total_revenue": {"$sum": "$total"},
            "order_count": {"$sum": 1},
            "avg_order_value": {"$avg": "$total"}
        }
    },
    # Stage 2: Sort by revenue
    {"$sort": {"total_revenue": -1}}
]

print("Revenue by Order Status:")
print("-" * 70)
for doc in db.orders.aggregate(pipeline):
    print(f"{doc['_id']:12} | Revenue: ${doc['total_revenue']:>10,.2f} | "
          f"Orders: {doc['order_count']:3} | Avg: ${doc['avg_order_value']:>8,.2f}")

Revenue by Order Status:
----------------------------------------------------------------------
shipped      | Revenue: $ 50,504.32 | Orders:  23 | Avg: $2,195.84
cancelled    | Revenue: $ 50,504.06 | Orders:  26 | Avg: $1,942.46
pending      | Revenue: $ 47,060.64 | Orders:  19 | Avg: $2,476.88
delivered    | Revenue: $ 28,317.66 | Orders:  17 | Avg: $1,665.74
processing   | Revenue: $ 10,291.19 | Orders:  15 | Avg: $  686.08


## Filter Then Aggregate

In [14]:
pipeline = [
    # Stage 1: Filter first (more efficient!)
    {"$match": {"status": "delivered"}},
    
    # Stage 2: Group by region
    {
        "$group": {
            "_id": "$region",
            "revenue": {"$sum": "$total"},
            "orders": {"$sum": 1}
        }
    },
    
    {"$sort": {"revenue": -1}}
]

print("Delivered Revenue by Region:")
for doc in db.orders.aggregate(pipeline):
    print(f"  {doc['_id']:10} | ${doc['revenue']:>10,.2f} | {doc['orders']} orders")

Delivered Revenue by Region:
  Asia       | $ 11,566.52 | 7 orders
  Americas   | $  8,389.69 | 3 orders
  Europe     | $  8,361.45 | 7 orders


## Working with Arrays: $unwind

The `$unwind` stage "flattens" arrays — creates one document per array element.

**Before $unwind:**
```json
{"_id": "ORD1", "items": [{"name": "A"}, {"name": "B"}]}
```

**After $unwind:**
```json
{"_id": "ORD1", "items": {"name": "A"}}
{"_id": "ORD1", "items": {"name": "B"}}
```

In [15]:
# Revenue by product category
pipeline = [
    {"$match": {"status": "delivered"}},
    
    # Flatten the items array
    {"$unwind": "$items"},
    
    # Group by product category
    {
        "$group": {
            "_id": "$items.category",
            "total_revenue": {"$sum": "$items.subtotal"},
            "units_sold": {"$sum": "$items.quantity"}
        }
    },
    
    {"$sort": {"total_revenue": -1}}
]

print("Category Performance (Delivered Orders):")
print("-" * 55)
for doc in db.orders.aggregate(pipeline):
    print(f"{doc['_id']:18} | Revenue: ${doc['total_revenue']:>10,.2f} | Units: {doc['units_sold']}")

Category Performance (Delivered Orders):
-------------------------------------------------------
Electronics        | Revenue: $ 14,309.56 | Units: 44
Furniture          | Revenue: $ 13,671.39 | Units: 61
Office Supplies    | Revenue: $    336.71 | Units: 29


## Top Products Analysis

In [16]:
pipeline = [
    {"$match": {"status": {"$in": ["shipped", "delivered"]}}},
    {"$unwind": "$items"},
    {
        "$group": {
            "_id": "$items.product_id",
            "product_name": {"$first": "$items.product_name"},
            "revenue": {"$sum": "$items.subtotal"},
            "units_sold": {"$sum": "$items.quantity"}
        }
    },
    {"$sort": {"revenue": -1}},
    {"$limit": 5}
]

print("Top 5 Products by Revenue:")
print("-" * 55)
for i, doc in enumerate(db.orders.aggregate(pipeline), 1):
    print(f"{i}. {doc['product_name']:25} | ${doc['revenue']:>10,.2f} | {doc['units_sold']} units")

Top 5 Products by Revenue:
-------------------------------------------------------
1. Laptop Pro 15             | $ 29,899.77 | 23 units
2. Monitor 27 inch           | $ 17,199.57 | 43 units
3. Standing Desk             | $ 15,399.72 | 28 units
4. Office Chair              | $  7,199.76 | 24 units
5. Mechanical Keyboard       | $  4,029.69 | 31 units


## Joining Collections: $lookup

In [17]:
pipeline = [
    {"$match": {"status": "delivered"}},
    
    # Join with customers collection
    {
        "$lookup": {
            "from": "customers",           # Collection to join
            "localField": "customer_id",   # Field in orders
            "foreignField": "_id",         # Field in customers
            "as": "customer"               # Output array field
        }
    },
    
    # Unwind the result (always 1 element for 1:1)
    {"$unwind": "$customer"},
    
    # Group by segment
    {
        "$group": {
            "_id": "$customer.segment",
            "revenue": {"$sum": "$total"},
            "orders": {"$sum": 1},
            "avg_order": {"$avg": "$total"}
        }
    },
    
    {"$sort": {"revenue": -1}}
]

print("Revenue by Customer Segment:")
for doc in db.orders.aggregate(pipeline):
    print(f"  {doc['_id']:10} | ${doc['revenue']:>10,.2f} | {doc['orders']} orders | Avg: ${doc['avg_order']:.2f}")

Revenue by Customer Segment:
  Premium    | $ 18,787.25 | 8 orders | Avg: $2348.41
  Standard   | $  5,771.67 | 6 orders | Avg: $961.95
  Budget     | $  3,758.74 | 3 orders | Avg: $1252.91


> ⚠️ **Performance Warning:** `$lookup` can be expensive at scale. Consider denormalization for frequent joins.

## Group Accumulators

| Accumulator | Purpose | Example |
|-------------|---------|--------|
| `$sum` | Sum values | `{"total": {"$sum": "$amount"}}` |
| `$avg` | Average | `{"avg_price": {"$avg": "$price"}}` |
| `$min`/`$max` | Min/Max value | `{"lowest": {"$min": "$price"}}` |
| `$first`/`$last` | First/Last in group | `{"first": {"$first": "$date"}}` |
| `$push` | Collect into array | `{"items": {"$push": "$name"}}` |
| `$addToSet` | Unique values only | `{"cats": {"$addToSet": "$cat"}}` |

> 💡 Use `_id: null` to aggregate the entire collection.

---
# Part 3: Data Modeling
---

## Embedding vs. Referencing

### Embedding (Denormalization)

**When to use:**
- 1:1 or 1:few relationships
- Data is always accessed together
- Data rarely changes

**Benefits:**
- Single query retrieval
- Atomic updates
- Better read performance

```json
// Order with embedded customer info
{
  "_id": "order_123",
  "customer": {
    "name": "John Smith",
    "email": "john@example.com"
  },
  "items": [...],
  "total": 149.95
}
```

### Referencing (Normalization)

**When to use:**
- 1:many or many:many relationships
- Data is accessed independently
- Large/growing subdocuments

**Benefits:**
- No data duplication
- No 16MB document limit issues
- Easier updates

```json
// Order with customer reference
{
  "_id": "order_123",
  "customer_id": "cust_456",
  "items": [...],
  "total": 149.95
}
```

> 💡 **Key Principle:** Model data according to **how your application accesses it**, not how it's structured logically.

## Schema Design Patterns

### Extended Reference Pattern
Store frequently-accessed fields from referenced document directly.

**Example:** Store `customer_id` AND `customer_name` in orders for display without lookup.

### Bucket Pattern
Group time-series data into buckets (e.g., hourly readings in one document).

**Benefits:** Fewer documents, pre-computed aggregates, efficient range queries.

---
# Part 4: Indexing
---

## Why Indexes Matter

| Without Index | With Index |
|---------------|------------|
| Collection Scan | B-Tree Lookup |
| Check **every** document | Jump directly to matches |
| O(n) — Slow! | O(log n) — Fast! |

**Trade-off:** Indexes speed up *reads* but slow down *writes* (index must be updated on every insert/update/delete).

In [18]:
# Single field index
db.orders.create_index([("status", ASCENDING)], name="status_idx")
print("Created index on 'status'")

# Compound index (field ORDER matters!)
db.orders.create_index([
    ("customer_id", ASCENDING),
    ("order_date", DESCENDING)
], name="customer_date_idx")
print("Created compound index on 'customer_id' + 'order_date'")

Created index on 'status'
Created compound index on 'customer_id' + 'order_date'


In [19]:
# List all indexes on the orders collection
print("Indexes on orders collection:")
for idx in db.orders.list_indexes():
    print(f"  {idx['name']}: {dict(idx['key'])}")

Indexes on orders collection:
  _id_: {'_id': 1}
  status_idx: {'status': 1}
  customer_date_idx: {'customer_id': 1, 'order_date': -1}


## The ESR Rule

Order compound index fields as:

1. **E**quality conditions first (exact matches)
2. **S**ort fields second
3. **R**ange conditions last (`$gt`, `$lt`, etc.)

**Example Query:**
```python
db.orders.find({"status": "active", "total": {"$gt": 100}}).sort("date", -1)
```

**Optimal Index:**
```python
{"status": 1, "date": -1, "total": 1}
#  Equality    Sort       Range
```

---
# Part 5: Schema Validation
---

In [20]:
# Create collection with validation rules
db.drop_collection("validated_customers")

db.create_collection("validated_customers", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["name", "email"],
        "properties": {
            "name": {
                "bsonType": "string",
                "minLength": 1,
                "description": "Customer name is required"
            },
            "email": {
                "bsonType": "string",
                "pattern": "^.+@.+\\..+$",
                "description": "Must be valid email"
            },
            "segment": {
                "enum": ["Budget", "Standard", "Premium", None]  # Python None → BSON null
            }
        }
    }
})

print("✓ Created validated_customers with schema validation")

✓ Created validated_customers with schema validation


In [21]:
# Test: Valid insert - should succeed
try:
    db.validated_customers.insert_one({
        "name": "Test User",
        "email": "test@example.com",
        "segment": "Standard"
    })
    print("✓ Valid document inserted successfully")
except Exception as e:
    print(f"✗ Insert failed: {e}")

✓ Valid document inserted successfully


In [22]:
# Test: Invalid insert - missing required field
try:
    db.validated_customers.insert_one({
        "email": "bad@example.com",
        "segment": "Budget"
        # Missing name!
    })
    print("✓ Document inserted (unexpected!)")
except Exception as e:
    print("✗ Validation failed as expected: missing name")

✗ Validation failed as expected: missing name


In [23]:
# Test: Invalid insert - bad email format
try:
    db.validated_customers.insert_one({
        "name": "Another User",
        "email": "not-an-email",
        "gdpr_consent": True
    })
    print("✓ Document inserted (unexpected!)")
except Exception as e:
    print("✗ Validation failed as expected: invalid email format")

✗ Validation failed as expected: invalid email format


---
# Part 6: Exercises
---

Complete the following exercises to practice what you've learned.

## Exercise 1: Complex Query

**Task:** Find all orders that:
- Have total > $200
- Status is "delivered" or "shipped"
- Contain at least one Electronics item

In [24]:
# YOUR CODE HERE



## Exercise 2: Regional Performance Pipeline

**Task:** Create a pipeline that shows performance by **region** for non-cancelled orders:

- Filter out cancelled orders
- Group by region
- Show: total revenue, number of orders, average order value, and the **highest single order** (`$max`)
- Sort by total revenue descending

*Hint: Use `$ne` to exclude cancelled status, and the `$max` accumulator for highest order.*

In [25]:
# YOUR CODE HERE
pipeline = [
    # Add your stages here
]



## Exercise 3: Top Products Pipeline

**Task:** Find the top 5 best-selling products by **units sold** (not revenue).

Only include shipped or delivered orders.

Output: product_id, product_name, units_sold

In [26]:
# YOUR CODE HERE
pipeline = [
    # Add your stages here
]



## Exercise 4: Per-Customer Revenue with $lookup

**Task:** Find the **top 5 customers** by total spending on delivered orders.

- Filter for delivered orders only
- Use `$lookup` to join with the customers collection
- Group by customer name and include their **city** and **segment**
- Show: total revenue and number of orders per customer
- Sort by total revenue descending, limit to 5

*Hint: After `$lookup` and `$unwind`, use `$first` to capture city and segment in `$group`.*

In [27]:
# YOUR CODE HERE
pipeline = [
    # Add your stages here
]



## Exercise 5: Create an Index

**Task:** Create an optimal compound index for this query pattern:

```python
db.orders.find({"status": "delivered", "total": {"$gt": 100}}).sort("order_date", -1)
```

*Hint: Apply the ESR rule*

In [28]:
# YOUR CODE HERE



## Exercise 6: Schema Validation

**Task:** Create a collection called `validated_products` with these rules:

- `name` is required (string, min length 1)
- `price` is required (number, must be positive)
- `category` must be one of: "Electronics", "Furniture", "Office Supplies"
- `stock` must be an integer ≥ 0

In [29]:
# YOUR CODE HERE



---
# Summary
---

## Key Takeaways

1. **Advanced Operators:** `$and`, `$or`, `$in`, `$elemMatch` enable complex queries

2. **Aggregation Pipeline:** `$match` → `$unwind` → `$group` → `$sort` → `$project`

3. **Data Modeling:** Embed for read performance, reference for flexibility

4. **Indexing:** Use ESR rule (Equality → Sort → Range)

5. **Schema Validation:** Enforce data quality at database level

6. **Performance:** Profile queries with `explain()` and optimize with indexes

## Quiz Preparation — Session 3

**Review these topics:**

- NoSQL types & use cases
- CAP theorem trade-offs
- CRUD operations syntax
- Query operators
- Aggregation stages
- Embedding vs. referencing
- Index types & ESR rule
- Schema validation

**Format:** 20 Multiple Choice | 20 Minutes | Sessions 1 & 2 Content